# Notebook 4 of 7: LSTM -- Better Memory

## Giving Our AI a Notebook and Pen

**Series: Understanding AI Through Italian Music**

---

In the previous notebook, we built our first AI -- a **Recurrent Neural Network (RNN)** -- and watched it learn to write Italian lyrics. It was exciting! The loss went down, and the model generated text that looked vaguely Italian.

But there was a problem: **the RNN's memory fades quickly.**

Imagine trying to write a song where you can only remember the last 5 words you wrote. You would lose track of the theme. You would forget the rhyme scheme. If the first line was about the sea, by the fourth line you might be writing about pizza -- because you have already forgotten the sea.

This is exactly what happens to an RNN. It reads one word at a time, and by the time it reaches the end of a sentence, the beginning has become fuzzy. For short phrases it does fine, but for anything longer, the meaning drifts away.

In **1997**, two German researchers named Sepp Hochreiter and Jurgen Schmidhuber published a paper with a solution. They called it the **LSTM** -- **Long Short-Term Memory**. It was designed from the ground up to solve this exact memory problem.

In this notebook, we will:

1. **Understand** what makes the LSTM different from the RNN
2. **Build** an LSTM model and train it on the same Italian lyrics
3. **Compare** the LSTM's output directly against the RNN
4. **See** why better memory leads to better writing

Let's give our AI a notebook and pen.

## What Makes LSTM Different?

Here is a simple analogy.

**The RNN is a student listening to a lecture with no notes.** It hears each word the professor says, tries to hold everything in its head, and gradually the earlier parts of the lecture slip away. By the end of class, it can remember the last few minutes clearly, but the opening? Gone.

**The LSTM is a student who brings a notebook and pen.** It still listens word by word, but now it has a system for deciding:

1. **What new information to write down** (the "input gate")
2. **What old notes to erase** (the "forget gate")
3. **What notes to share when asked a question** (the "output gate")

These three gates work together to create a **long-term memory** that travels alongside the word-by-word reading. The LSTM can choose to remember that the song is about the sea, even as it processes dozens of words in between.

Here is a simple text diagram comparing the two:

```
RNN (no notebook):
                                         memory fades...
  Word 1  -->  Word 2  -->  Word 3  -->  Word 4  -->  Word 5
  [mare]      [blu]        [grande]     [???]        [???]
   |            |             |            |            |
   v            v             v            v            v
  Memory --> Memory -----> Memory ----> Memory ----> Memory
  (strong)   (OK)         (weak)      (very weak)  (almost gone)


LSTM (with notebook):
                                         memory preserved!
  Word 1  -->  Word 2  -->  Word 3  -->  Word 4  -->  Word 5
  [mare]      [blu]        [grande]     [il]         [sole]
   |            |             |            |            |
   v            v             v            v            v
  Memory --> Memory -----> Memory ----> Memory ----> Memory
  (strong)   (strong)     (strong)    (strong)     (strong)
              |             |            |            |
          "write down   "keep the    "this word    "share the
           the color"    sea notes"   isn't key,    sea theme
                                      skip it"      for output"
```

The cost of this better memory? **More parameters.** The gates are made of extra numbers that the model needs to learn. An LSTM is roughly **4 times larger** than an RNN with the same dimensions, because each gate needs its own set of weights.

More parameters means more to learn -- but also more capacity to remember.

## Setup

Run the cell below to load our tools. This is the same setup as Notebook 3, except we are importing **LSTMModel** in addition to RNNModel -- so we can compare them head-to-head later.

You do not need to understand every line. Just run it and move on.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer

# Our custom modules
from src.dataset import ItalianLyricsDataset, load_lyrics
from src.models import RNNModel, LSTMModel, model_summary
from src.training import train_model, get_optimizer
from src.generation import generate_rnn_lstm, generate_greedy
from src.visualization import plot_training_loss

# Show charts inside the notebook
%matplotlib inline

print("All tools loaded successfully!")

## Load Data and Build the Model

We will use the **exact same data and settings** as Notebook 3 -- 500 Italian songs, the same tokenizer, the same batch size, the same dimensions. This way, any difference in the results is purely because of the architecture change from RNN to LSTM.

The only difference: when we look at the model summary, you will see the LSTM has **more parameters** than the RNN. Those extra parameters are the gates -- the notebook and pen that help it remember.

In [ ]:
# Load 500 Italian song lyrics -- same as Notebook 3
lyrics = load_lyrics('../data/italian_lyrics.txt', max_songs=500)
print(f"Loaded {len(lyrics)} songs")

# Set up the tokenizer (word-to-number dictionary)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Create the dataset and dataloader -- same settings as before
MAX_LENGTH = 128   # Maximum number of tokens per song
BATCH_SIZE = 32    # How many songs to process at once

dataset = ItalianLyricsDataset(lyrics, tokenizer, max_length=MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Vocabulary size: {len(tokenizer):,} tokens")
print(f"Batches per epoch: {len(dataloader)}")

# ---------- Build the LSTM model ----------
device = torch.device('cpu')
VOCAB_SIZE = len(tokenizer)
EMBEDDING_DIM = 256
HIDDEN_DIM = 512

lstm_model = LSTMModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
).to(device)

# Show the model summary
print("\n--- Model size ---")
lstm_params = model_summary(lstm_model, name="LSTM")

# For comparison, build a quick RNN to see the difference
temp_rnn = RNNModel(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM)
rnn_params = model_summary(temp_rnn, name="RNN (for comparison)")

ratio = lstm_params / rnn_params
print(f"\nThe LSTM is {ratio:.1f}x larger than the RNN.")
print("The extra parameters are the gates -- the notebook and pen that help it remember.")

## Train the LSTM

Training works exactly the same way as Notebook 3:

1. Show the model a batch of songs
2. Ask it to predict the next word at every position
3. Measure how wrong it was (the **loss**)
4. Adjust the model's numbers to be slightly less wrong
5. Repeat

We will train for **3 epochs** (three full passes through all 500 songs). Watch the loss go down -- the model is learning!

Because the LSTM has more parameters than the RNN, it may take a bit longer per epoch. That is the trade-off: more memory capacity costs more computation.

In [ ]:
EPOCHS = 3

# Create the optimizer (controls how the model adjusts its numbers)
lstm_optimizer = get_optimizer(lstm_model)

print(f"Training LSTM for {EPOCHS} epochs on {len(lyrics)} songs...")
print(f"This may take a few minutes.\n")

lstm_history = train_model(
    model=lstm_model,
    dataloader=dataloader,
    optimizer=lstm_optimizer,
    device=device,
    epochs=EPOCHS,
    model_name="LSTM"
)

print(f"\nDone! Final loss: {lstm_history['epoch_losses'][-1]:.4f}")

## Generate Lyrics with the LSTM

Now for the fun part -- let's see what our trained LSTM writes!

We will try two generation methods:

- **Greedy** -- always picks the single most likely next word. This tends to produce repetitive, "safe" text. Think of it as a very cautious writer who never takes risks.
- **Sampling (top-k / top-p)** -- picks from the most likely words with some randomness. This produces more varied and creative text. Think of it as a writer who sometimes surprises you.

We saw both of these in Notebook 3 with the RNN. Let's see if the LSTM does better.

In [ ]:
seed_text = "Amore mio"

# --- Greedy generation (always pick the most likely word) ---
print("=" * 60)
print("  LSTM -- Greedy Generation")
print("  (always picks the most likely next word)")
print("=" * 60)
greedy_output = generate_greedy(lstm_model, tokenizer, seed_text, max_length=60)
print(f"\n  Seed: \"{seed_text}\"\n")
print(f"  {greedy_output}")

# --- Sampling generation (creative, with randomness) ---
print("\n" + "=" * 60)
print("  LSTM -- Sampling Generation (top-k + top-p)")
print("  (picks from likely words with some randomness)")
print("=" * 60)

for i in range(3):
    sampled_output = generate_rnn_lstm(
        lstm_model, tokenizer, seed_text,
        max_length=60, temperature=0.8, top_k=50, top_p=0.9
    )
    print(f"\n  Sample {i+1}: {sampled_output}")

print("\n" + "=" * 60)
print("\nNotice: greedy output tends to be repetitive,")
print("while sampling produces different text each time.")

## Side-by-Side: RNN vs LSTM

Seeing the LSTM's output on its own is interesting, but the real test is a **direct comparison**. Let's train an RNN with the exact same data, the same settings, and the same number of epochs. Then we will generate from both models using the same seed text.

This is how scientists compare approaches: change **one thing** (the architecture) and keep everything else identical. Any difference in the output must be due to the LSTM's memory gates.

In [ ]:
# Build and train an RNN with identical settings
rnn_model = RNNModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
).to(device)

rnn_optimizer = get_optimizer(rnn_model)

print(f"Training RNN for {EPOCHS} epochs (same settings as the LSTM)...\n")

rnn_history = train_model(
    model=rnn_model,
    dataloader=dataloader,
    optimizer=rnn_optimizer,
    device=device,
    epochs=EPOCHS,
    model_name="RNN"
)

print(f"\nRNN final loss:  {rnn_history['epoch_losses'][-1]:.4f}")
print(f"LSTM final loss: {lstm_history['epoch_losses'][-1]:.4f}")

Now let's generate from both models using the same seed text and compare the results side by side.

In [ ]:
# Generate from both models with the same seed
seed_texts = ["Amore mio", "La notte", "Non posso"]

for seed in seed_texts:
    print("=" * 60)
    print(f"  Seed: \"{seed}\"")
    print("=" * 60)

    # Set the same random seed for fair comparison
    torch.manual_seed(42)
    rnn_output = generate_rnn_lstm(
        rnn_model, tokenizer, seed,
        max_length=60, temperature=0.8, top_k=50, top_p=0.9
    )

    torch.manual_seed(42)
    lstm_output = generate_rnn_lstm(
        lstm_model, tokenizer, seed,
        max_length=60, temperature=0.8, top_k=50, top_p=0.9
    )

    print(f"\n  RNN:  {rnn_output}")
    print(f"\n  LSTM: {lstm_output}")
    print()

print("=" * 60)
print("\nLook for differences in:")
print("  - Coherence: does the text stay on topic?")
print("  - Repetition: does it get stuck in loops?")
print("  - Italian quality: do the words flow naturally?")

## Comparing Training

Numbers are helpful, but a picture tells the story more clearly. The chart below shows how the loss (wrongness score) decreased for both models during training.

- **Red line** = RNN
- **Blue line** = LSTM
- **Lower is better** -- the model is making fewer mistakes

If the LSTM line is below the RNN line, it means the LSTM learned to predict Italian words more accurately.

In [ ]:
import matplotlib.pyplot as plt

# Plot both training curves on one chart
fig = plot_training_loss(
    [rnn_history, lstm_history],
    title="Training Comparison: RNN vs LSTM"
)
plt.show()

# Print a summary table
print("\n--- Training Summary ---")
print(f"{'':>10} {'Final Loss':>12} {'Training Time':>15}")
print("-" * 40)
print(f"{'RNN':>10} {rnn_history['epoch_losses'][-1]:>12.4f} {rnn_history['training_time']/60:>13.1f} min")
print(f"{'LSTM':>10} {lstm_history['epoch_losses'][-1]:>12.4f} {lstm_history['training_time']/60:>13.1f} min")

## The Verdict

The LSTM is better at maintaining coherence over longer sequences. Its memory gates let it hold onto important context -- like the theme of a song or a rhyme pattern -- even as it processes many words in between.

**But both models still have fundamental limitations:**

- They **read one word at a time**, left to right. This is slow, and it means they can never "look ahead" to see what is coming.
- They still **struggle with very long-range dependencies**. Even the LSTM's notebook has limited pages -- if a song is hundreds of words long, early information can still fade.
- They **produce text that lacks deep understanding**. The models learn statistical patterns ("this word often follows that word") but do not truly "understand" what the words mean. A song about love and a song about loss might use similar word patterns, and the model does not grasp the emotional difference.

These are not just limitations of our small models. They are fundamental limitations of **sequential processing** -- reading one thing at a time. To truly overcome them, we need a completely different approach.

And that is exactly what happened in 2017.

## Real-World Context

LSTMs were not just a research curiosity. They were the **state of the art** in language AI for several years, roughly **2014 to 2017**. During that period, LSTMs powered real products that millions of people used every day:

- **Google Translate** -- Early neural machine translation systems used LSTMs to read a sentence in one language and generate it in another. If you used Google Translate between 2016 and 2018, an LSTM was doing the heavy lifting.

- **Siri and Google Assistant** -- The language understanding component (figuring out what you meant when you said "set a timer for 10 minutes") relied on LSTM networks.

- **Autocomplete on your phone** -- When your keyboard predicts the next word you are about to type, early versions of that feature used LSTMs -- very similar to what we just built, but trained on billions of text messages instead of 500 Italian songs.

- **Speech recognition** -- Converting spoken words into text was one of the LSTM's greatest strengths, since speech is naturally sequential.

Then in **2017**, a team of researchers at Google published a paper with a provocative title: **"Attention Is All You Need."** That paper introduced the **Transformer** architecture, and it changed everything.

The Transformer did not read one word at a time. It read the **entire sentence at once**. And that single change made it so much more powerful that within a few years, LSTMs were largely replaced in cutting-edge AI systems.

The Transformer is the architecture behind GPT, ChatGPT, Claude, and virtually every modern AI language system. We will meet it in the next notebook.

## Key Takeaways

Here is what we learned in this notebook:

| Concept | What It Means |
|---|---|
| **LSTM** | Long Short-Term Memory -- an improved RNN with memory gates |
| **Gates** | Three mechanisms (input, forget, output) that control what the model remembers and forgets |
| **More parameters** | The LSTM is roughly 4x larger than an RNN with the same dimensions, because the gates need extra numbers |
| **Better long-range memory** | The LSTM can hold onto earlier information better than the RNN, producing more coherent text |
| **Still sequential** | Like the RNN, the LSTM reads one word at a time -- it cannot look at the whole sentence at once |
| **State of the art (2014-2017)** | LSTMs powered Google Translate, Siri, phone autocomplete, and speech recognition |
| **Replaced by Transformers** | The 2017 "Attention Is All You Need" paper introduced a fundamentally better approach |

**The big picture so far:**

```
Notebook 2: Tokenization     -- how AI reads text (words to numbers)
Notebook 3: RNN              -- first model, learns patterns, but memory fades
Notebook 4: LSTM             -- better memory with gates, but still reads one word at a time
Notebook 5: Transformer      -- reads everything at once (coming next!)
```

## What's Next

We have now seen two approaches to teaching AI to write: the RNN (a student with no notes) and the LSTM (a student with a notebook). Both read one word at a time, and both have limits on how far back they can remember.

**In the next notebook, we meet the Transformer** -- the architecture that powers ChatGPT, Claude, and changed AI forever. Instead of reading one word at a time, the Transformer reads the entire sentence at once using a mechanism called **attention**. It was such a leap forward that it made both RNNs and LSTMs largely obsolete for language tasks within just a few years.

See you in **Notebook 5: The Transformer**.

---

*Notebook 4 of 7 -- Understanding AI Through Italian Music*